# M17 · Jacobienne et règle de la chaîne — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m17_jacobienne/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m17_jacobienne/03_notebook.ipynb`, créée par `python tools/start_chapter.py M17`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M17.1 à M17.7, M17.11 | Les exercices papier (vérification) | ✏️ | | |
| A | M17.14 | La jacobienne par différences centrées, colonne par colonne · `numerical_jacobian` | 🔨 | ★★ | 20 |
|  | M17.15 | Une jacobienne diagonale avec une seule paire d'évaluations · `diagonal_jacobian` | 🔨 | ★ | 15 |
|  | M17.16 | Un produit jacobienne-vecteur sans former la jacobienne · `jacobian_vector_product` | 🔨 | ★ | 15 |
| B | M17.17 | `sympy.Matrix.jacobian` et `scipy.differentiate.jacobian` contre ta jacobienne numérique | 📦 | ★ | 15 |
|  | M17.18 | Prédire la forme et les zéros de la jacobienne d'une composée, puis vérifier | 🔮 | ★ | 10 |
|  | M17.19 | Une longue composition : une dérivée qui s'éteint ou explose selon la pente au point fixe | 🔬 | ★★ | 20 |
|  | M17.20 | Une grille du plan déformée par une application non linéaire, et par son approximation linéaire | 🎨 | ★ | 20 |
| C | M17.21 | Chasse à l'erreur : une jacobienne transposée, un produit dans le mauvais ordre, une jacobienne évaluée au mauvais point | 🐛 | ★ | 15 |
|  | M17.22 | Un test de propriété : J_{g∘f} = J_g J_f sur des applications tirées au hasard | 🛠️ | ★ | 15 |
|  | M17.23 | Défi : la jacobienne de l'inversion X ↦ X⁻¹, vue comme une application de ℝ^{n²} dans ℝ^{n²} | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M17")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : la jacobienne des coordonnées polaires et l'approximation linéaire, une jacobienne diagonale, une composée de trois applications, la normalisation $\mathbf{x}/\lVert\mathbf{x}\rVert$, un gradient dans les deux conventions, des dérivées à gauche et à droite et une composée de sept copies, deux lois de Boltzmann.
- Écrire les trois fonctions de `mymath/jacobian.py` et les tester.
- Comparer ta jacobienne à celles de SymPy et de SciPy, prédire la jacobienne d'une composée, suivre la dérivée d'une longue itération, voir une approximation linéaire, corriger des erreurs classiques, écrire un test de propriété, et relever un défi : la jacobienne de l'inversion des matrices.

**Rappel express.** Pour $\mathbf{f} : \mathbb{R}^n \to \mathbb{R}^m$, $\mathbf{J}_{\mathbf{f}}$ a la forme $(m, n)$, $J_{ij} = \frac{\partial f_i}{\partial x_j}$ : une ligne par sortie, une colonne par entrée. $\mathbf{f}(\mathbf{x} + \mathbf{h}) \approx \mathbf{f}(\mathbf{x}) + \mathbf{J}\mathbf{h}$, et $\lvert\det\mathbf{J}\rvert$ multiplie les aires. Coordonnée par coordonnée : $\mathbf{J} = \operatorname{diag}(g'(x_1), \ldots, g'(x_n))$, $\mathbf{J}\mathbf{v} = g'(\mathbf{x}) \odot \mathbf{v}$. $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}(\mathbf{x}) = \mathbf{J}_{\mathbf{g}}(\mathbf{f}(\mathbf{x}))\,\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ : la dernière à gauche, au point intermédiaire. $\nabla_{\mathbf{x}}(L \circ \mathbf{f}) = \nabla_{\mathbf{y}}L\,\mathbf{J}_{\mathbf{f}}$ en ligne, $\mathbf{J}_{\mathbf{f}}^\top\nabla_{\mathbf{y}}L$ de la forme de la variable. $\mathbf{J}\mathbf{v} \approx \frac{\mathbf{f}(\mathbf{x} + h\mathbf{v}) - \mathbf{f}(\mathbf{x} - h\mathbf{v})}{2h}$ : deux évaluations. Une itération : $\frac{\mathrm{d}x_L}{\mathrm{d}x_0} = \prod_k\varphi'(x_k)$. Boltzmann : $p_i = \frac{e^{-E_i/T}}{\sum_je^{-E_j/T}}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M17.1 à M17.7, M17.11)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ;
- une matrice comme la liste de ses **lignes** : `[[1, 0], [-2, 3]]` ; un nombre décimal s'y écrit avec un point, `[[0.5, 1], [0, 2]]`, ou comme un calcul, `[[1/2, 1], [0, 2]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit.

Les réponses pas encore remplies affichent ⏳.

**Ex M17.1 — La jacobienne des coordonnées polaires, son déterminant, et l'approximation linéaire près d'un point**

In [ ]:
# Ex M17.1: the values you found on paper
answer_M17_1a = ...  # a) a list of 2 numbers (4 decimals)
answer_M17_1b = ...  # b) a matrix (list of rows, 4 decimals)
answer_M17_1c = ...  # c) a number
answer_M17_1d = ...  # d) a list of 2 numbers (4 decimals)
answer_M17_1e = ...  # e) a list of 2 numbers (4 decimals)
answer_M17_1f = ...  # f) a number (4 decimals)
answer_M17_1g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M17_1a, answer_M17_1b, answer_M17_1c, answer_M17_1d, answer_M17_1e, answer_M17_1f, answer_M17_1g]):
    wbm.check(f"M17.1{letter}", answer)

**Ex M17.2 — Une fonction appliquée coordonnée par coordonnée : la jacobienne diagonale, et son produit par un vecteur**

In [ ]:
# Ex M17.2: the values you found on paper
answer_M17_2a = ...  # a) a list of 4 numbers
answer_M17_2b = ...  # b) a list of 4 numbers
answer_M17_2c = ...  # c) an integer
answer_M17_2d = ...  # d) an integer
answer_M17_2e = ...  # e) a list of 2 integers
answer_M17_2f = ...  # f) a list of 4 numbers (3 decimals)

for letter, answer in zip("abcdef", [answer_M17_2a, answer_M17_2b, answer_M17_2c, answer_M17_2d, answer_M17_2e, answer_M17_2f]):
    wbm.check(f"M17.2{letter}", answer)

**Ex M17.3 — Dériver une composée de trois applications par le produit de leurs jacobiennes**

In [ ]:
# Ex M17.3: the values you found on paper
answer_M17_3a = ...  # a) a list of 3 numbers
answer_M17_3b = ...  # b) a matrix (list of rows)
answer_M17_3c = ...  # c) a matrix (list of rows)
answer_M17_3d = ...  # d) a matrix (one row: [[a, b]])
answer_M17_3e = ...  # e) a list of 2 numbers
answer_M17_3f = ...  # f) a list of 2 integers

for letter, answer in zip("abcdef", [answer_M17_3a, answer_M17_3b, answer_M17_3c, answer_M17_3d, answer_M17_3e, answer_M17_3f]):
    wbm.check(f"M17.3{letter}", answer)

**Ex M17.4 — La normalisation x/‖x‖ : sa jacobienne (I − uuᵀ)/‖x‖, et la direction qu'elle écrase**

In [ ]:
# Ex M17.4: the values you found on paper
answer_M17_4a = ...  # a) a list of 4 numbers
answer_M17_4b = ...  # b) a matrix (list of rows)
answer_M17_4c = ...  # c) a list of 3 numbers
answer_M17_4d = ...  # d) a list of 3 numbers
answer_M17_4e = ...  # e) an integer
answer_M17_4f = ...  # f) a list of 3 numbers

for letter, answer in zip("abcdef", [answer_M17_4a, answer_M17_4b, answer_M17_4c, answer_M17_4d, answer_M17_4e, answer_M17_4f]):
    wbm.check(f"M17.4{letter}", answer)

**Ex M17.5 — Le gradient d'une fonction scalaire d'une composée, dans les deux conventions : J_fᵀ ∇L**

In [ ]:
# Ex M17.5: the values you found on paper
answer_M17_5a = ...  # a) a list of 3 numbers
answer_M17_5b = ...  # b) a list of 3 numbers
answer_M17_5c = ...  # c) a matrix (list of rows)
answer_M17_5d = ...  # d) a list of 2 numbers
answer_M17_5e = ...  # e) a list of 2 integers
answer_M17_5f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M17_5a, answer_M17_5b, answer_M17_5c, answer_M17_5d, answer_M17_5e, answer_M17_5f]):
    wbm.check(f"M17.5{letter}", answer)

**Ex M17.6 — Dérivées à gauche et à droite d'une fonction définie par morceaux, et la dérivée d'une composée de sept copies d'une fonction lisse**

In [ ]:
# Ex M17.6: the values you found on paper
answer_M17_6a = ...  # a) a number
answer_M17_6b = ...  # b) a list of 2 numbers
answer_M17_6c = ...  # c) a list of 2 numbers
answer_M17_6d = ...  # d) a list of 2 numbers
answer_M17_6e = ...  # e) an irreducible fraction, as text
answer_M17_6f = ...  # f) an irreducible fraction, as text
answer_M17_6g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M17_6a, answer_M17_6b, answer_M17_6c, answer_M17_6d, answer_M17_6e, answer_M17_6f, answer_M17_6g]):
    wbm.check(f"M17.6{letter}", answer)

**Ex M17.7 — Une loi de Boltzmann à quatre niveaux d'énergie : rapports de populations, énergie moyenne, et la température où le niveau fondamental garde la moitié**

In [ ]:
# Ex M17.7: the values you found on paper
answer_M17_7a = ...  # a) a number (4 decimals)
answer_M17_7b = ...  # b) a number (4 decimals)
answer_M17_7c = ...  # c) a number (4 decimals)
answer_M17_7d = ...  # d) a list of 2 numbers
answer_M17_7e = ...  # e) a number (4 decimals)
answer_M17_7f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M17_7a, answer_M17_7b, answer_M17_7c, answer_M17_7d, answer_M17_7e, answer_M17_7f]):
    wbm.check(f"M17.7{letter}", answer)

**Ex M17.11 — Le recuit simulé : une loi de Boltzmann sur cinq configurations, dont on baisse la température**

In [ ]:
# Ex M17.11: the values you found on paper
answer_M17_11a = ...  # a) a number (4 decimals)
answer_M17_11b = ...  # b) a number (2 decimals)
answer_M17_11c = ...  # c) a number (2 decimals)
answer_M17_11d = ...  # d) a number (4 decimals)
answer_M17_11e = ...  # e) a number (4 decimals)
answer_M17_11f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M17_11a, answer_M17_11b, answer_M17_11c, answer_M17_11d, answer_M17_11e, answer_M17_11f]):
    wbm.check(f"M17.11{letter}", answer)

## Partie A · Ta librairie : la jacobienne numérique, sa diagonale, et un produit jacobienne-vecteur

Fiche §1, §2 et §5. Les trois fonctions de `mymath/jacobian.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to C)
import inspect
import math
import time
import warnings

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import sympy

from wbm.nbtools import StopCheck, checked, drawn, fr, run_pytest_group, verdict

TEST_M17 = "tests/test_m17_jacobian.py"          # the tests of mymath/jacobian.py

### Ex M17.14 — La jacobienne par différences centrées, colonne par colonne · `numerical_jacobian` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la jacobienne d'une application de $\mathbb{R}^n$ dans $\mathbb{R}^m$ par différences centrées, et lire ses colonnes sur une carte de la Terre.  
**Prérequis :** fiche §1 ; M05a · **mymath :** `jacobian.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/jacobian.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis, mais ni SymPy ni `scipy.differentiate` (les tests comparent tes résultats aux leurs, et un test vérifie que ton module ne les importe pas). Ne modifie jamais le `x` reçu : travaille sur une copie en flottants, `np.array(x, dtype=float)` (une liste d'entiers doit être acceptée). Une valeur de `f` qui est un nombre compte comme un tableau de longueur 1 (`np.atleast_1d`). Les matrices et les vecteurs se rendent en tableaux NumPy de flottants.

Écris `numerical_jacobian(f, x, h=1e-5)` (fiche §1), en suivant sa docstring : `ValueError` si `x` n'est pas un tableau 1-D non vide, si `h` n'est pas un nombre $> 0$, ou si les valeurs de `f` ne sont pas des tableaux 1-D, toutes de la même longueur ; sinon la matrice $(m, n)$ dont la colonne $j$ est
$$\frac{\mathbf{f}(\mathbf{x} + h\,\mathbf{e}_j) - \mathbf{f}(\mathbf{x} - h\,\mathbf{e}_j)}{2h},$$
où $\mathbf{e}_j$ est le $j$-ième vecteur de la base canonique. La colonne $j$ est la vitesse de $\mathbf{f}(\mathbf{x})$ quand seule $x_j$ varie (fiche §1) ; la ligne $i$ est le gradient de la sortie $f_i$. La matrice entière coûte $2n$ évaluations de $\mathbf{f}$.

**L'usage.** On prend la Terre pour une sphère de rayon $R = 6\,371$ km. Le point de latitude $\varphi$ et de longitude $\lambda$ (en radians) est
$$\mathbf{S}(\varphi, \lambda) = R\,(\cos\varphi\cos\lambda,\ \cos\varphi\sin\lambda,\ \sin\varphi),$$
une application de $\mathbb{R}^2$ dans $\mathbb{R}^3$ (cellule fournie : `sphere_14`, et Lyon, $\varphi = 45{,}76°$ et $\lambda = 4{,}84°$, convertis en radians dans `LYON_14`). La colonne 1 de $\mathbf{J}_{\mathbf{S}}$ dit comment le point bouge (en km par radian) quand la latitude bouge, la colonne 2 quand la longitude bouge. Avec ta fonction, calcule la jacobienne $(3, 2)$ de $\mathbf{S}$ à Lyon, puis :
- a) la longueur, en km, d'un degré de longitude à cette latitude : la norme de la colonne 2, fois $\frac{\pi}{180}$ (2 décimales) ;
- b) la longueur, en km, d'un degré de latitude (2 décimales).

Dans ta copie : écris $\mathbf{J}_{\mathbf{S}}(\varphi, \lambda)$ à la main, et montre que la colonne 1 a la norme $R$ et la colonne 2 la norme $R\cos\varphi$. Pourquoi a) vaut-elle $R\cos\varphi\,\frac{\pi}{180}$ (sur quel cercle le point se déplace-t-il quand seule la longitude bouge, et quel est son rayon) ? Que vaudrait a) à l'équateur, et au pôle ? Combien d'évaluations de $\mathbf{S}$ ta fonction a-t-elle faites ?

In [ ]:
EARTH_RADIUS_14 = 6371.0                       # km: the Earth taken as a sphere
LYON_14 = np.radians([45.76, 4.84])            # Lyon: (latitude φ, longitude λ), converted into radians


def sphere_14(p):
    """S(φ, λ): the point of latitude p[0] and longitude p[1] (radians) of the sphere, in km."""
    phi, lam = p
    return EARTH_RADIUS_14 * np.array([np.cos(phi) * np.cos(lam), np.cos(phi) * np.sin(lam), np.sin(phi)])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M17")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M17, "numerical_jacobian or module_uses")

**L'usage** : la jacobienne de la sphère à Lyon, et ses deux colonnes.

In [ ]:
answer_M17_14a = answer_M17_14b = ...
with wbm.attempt("M17.14"):          # if you call numerical_jacobian here: ⏳ while it is not written
    jacobian_14 = ...     # the (3, 2) Jacobian of sphere_14 at LYON_14, by mymath.jacobian.numerical_jacobian
    answer_M17_14a = ...  # a) a number (2 decimals): the length of one degree of longitude, in km
    answer_M17_14b = ...  # b) a number (2 decimals): the length of one degree of latitude, in km

In [ ]:
wbm.check("M17.14a", answer_M17_14a, computed=True)
wbm.check("M17.14b", answer_M17_14b, computed=True)

### Ex M17.15 — Une jacobienne diagonale avec une seule paire d'évaluations · `diagonal_jacobian` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la jacobienne diagonale d'une fonction appliquée coordonnée par coordonnée avec deux évaluations, et comparer son coût à celui de la jacobienne entière.  
**Prérequis :** fiche §2 ; Ex M17.14 · **mymath :** `jacobian.py` · **Parcours :** C, E

Écris `diagonal_jacobian(f, x, h=1e-5)` (fiche §2), en suivant sa docstring : `ValueError` si `x` n'est pas un tableau 1-D non vide, si `h` n'est pas un nombre $> 0$, ou si $\mathbf{f}(\mathbf{x})$ n'a pas la longueur de $\mathbf{x}$ ; sinon la matrice diagonale $(n, n)$ de la jacobienne d'une fonction appliquée coordonnée par coordonnée. Puisque la sortie $i$ ne dépend que de $x_i$, on **bouge toutes les coordonnées à la fois** : la coordonnée $i$ de $\frac{\mathbf{f}(\mathbf{x} + h\mathbf{1}) - \mathbf{f}(\mathbf{x} - h\mathbf{1})}{2h}$ est la dérivée $g'(x_i)$, et `np.diag` en fait la matrice. Deux évaluations de $\mathbf{f}$ en tout. La fonction ne vérifie pas que $\mathbf{f}$ est appliquée coordonnée par coordonnée : sur une autre application, elle rend une diagonale fausse (un test le vérifie).

**L'usage.** $g = \arctan$, appliquée aux 300 points `X_15`, régulièrement espacés de $-3$ à $3$. La cellule fournie donne aussi `Counted` : `Counted(np.arctan)` se comporte comme `np.arctan` et compte ses appels dans son attribut `calls` (crée un nouveau compteur pour chaque appel de tes fonctions). Avec tes fonctions, calcule :
- a) le nombre d'évaluations de $\mathbf{f}$ faites par `diagonal_jacobian`, puis par `numerical_jacobian`, sur `X_15` (une liste de 2 entiers) ;
- b) la trace de la jacobienne de $\arctan$ en `X_15` (4 décimales) ;
- c) vrai ou faux : la plus grande différence, en valeur absolue, entre la diagonale de `numerical_jacobian` et celle de `diagonal_jacobian` est sous $10^{-8}$.

Dans ta copie : pourquoi la trace vaut-elle $\sum_i\frac{1}{1 + x_i^2}$ ? Les 300 points sont espacés d'environ $\frac{6}{300}$ : compare b) à la valeur approchée $\frac{300}{6}\int_{-3}^{3}\frac{\mathrm{d}t}{1 + t^2}$. Pourquoi les deux diagonales de c) sont-elles si proches (quels nombres chaque fonction calcule-t-elle pour la coordonnée $i$) ? Combien de coefficients a la matrice de `numerical_jacobian`, et combien sont non nuls ?

In [ ]:
X_15 = np.linspace(-3.0, 3.0, 300)       # 300 points evenly spaced from -3 to 3


class Counted:
    """A map that counts its calls: Counted(g)(p) is g(p), and .calls is the number of calls made so far."""

    def __init__(self, g):
        self.g, self.calls = g, 0

    def __call__(self, p):
        self.calls += 1
        return self.g(p)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M17")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M17, "diagonal_jacobian")

**L'usage** : la diagonale de la jacobienne de arctan, et ce qu'elle coûte.

In [ ]:
answer_M17_15a = answer_M17_15b = answer_M17_15c = ...
with wbm.attempt("M17.15"):          # if you call your functions here: ⏳ while they are not written
    answer_M17_15a = ...  # a) a list of 2 integers: the evaluations of f by diagonal_jacobian, then by numerical_jacobian
    answer_M17_15b = ...  # b) a number (4 decimals): the trace of the Jacobian of arctan at X_15
    answer_M17_15c = ...  # c) True or False

In [ ]:
for letter, answer in zip("abc", [answer_M17_15a, answer_M17_15b, answer_M17_15c]):
    wbm.check(f"M17.15{letter}", answer, computed=True)

### Ex M17.16 — Un produit jacobienne-vecteur sans former la jacobienne · `jacobian_vector_product` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer un produit jacobienne-vecteur avec deux évaluations, sans former la jacobienne, et mesurer ce qu'il économise.  
**Prérequis :** fiche §5 ; Ex M17.14 · **mymath :** `jacobian.py` · **Parcours :** C

Écris `jacobian_vector_product(f, x, v, h=1e-5)` (fiche §5), en suivant sa docstring : `ValueError` si `x` ou `v` n'est pas un tableau 1-D, s'ils sont vides ou de longueurs différentes, ou si `h` n'est pas un nombre $> 0$ ; sinon les $m$ coordonnées de
$$\mathbf{J}_{\mathbf{f}}(\mathbf{x})\,\mathbf{v} \approx \frac{\mathbf{f}(\mathbf{x} + h\mathbf{v}) - \mathbf{f}(\mathbf{x} - h\mathbf{v})}{2h},$$
la dérivée de toutes les sorties quand on part de $\mathbf{x}$ dans la direction $\mathbf{v}$ : deux évaluations de $\mathbf{f}$, quel que soit $n$, sans former la matrice.

**L'usage.** $\mathbf{f}(\mathbf{x})_i = x_ix_{i+1}$, les indices pris modulo $n$ (la dernière coordonnée est multipliée par la première) : `neighbours_16`, sur $n = 2\,000$ coordonnées, au point `X_16` (2 000 points régulièrement espacés de $-1$ à $1$) et dans la direction `V_16` ($\cos$ de 2 000 points régulièrement espacés de $0$ à $2\pi$). Avec tes fonctions, et `Counted` (M17.15), calcule :
- a) la norme euclidienne de $\mathbf{J}\mathbf{v}$ (2 décimales) ;
- b) le nombre d'évaluations de $\mathbf{f}$ faites par `jacobian_vector_product`, puis par `numerical_jacobian` (pour former $\mathbf{J}$, avant le produit `J @ v`) : une liste de 2 entiers.

Pour b), ta `numerical_jacobian` forme une matrice de 4 millions de coefficients : une fraction de seconde si tu remplis une matrice préparée par `np.zeros`, ou si tu empiles les colonnes une seule fois à la fin. Si elle prend plus de quelques secondes, compte ses évaluations sur les 200 premières coordonnées, et déduis b) de ce que coûte une colonne.

Dans ta copie : écris $(\mathbf{J}\mathbf{v})_i = v_ix_{i+1} + x_iv_{i+1}$ (quelles cases de la ligne $i$ de $\mathbf{J}$ sont non nulles ?), et compte les multiplications de cette formule, contre $n^2$ pour `J @ v` avec la matrice formée. Combien de coefficients de $\mathbf{J}$ sont non nuls ? Pourquoi la formule des différences donne-t-elle ici $\mathbf{J}\mathbf{v}$ exactement, aux arrondis près (de quel degré est chaque sortie) ?

In [ ]:
N_16 = 2000
X_16 = np.linspace(-1.0, 1.0, N_16)
V_16 = np.cos(np.linspace(0.0, 2 * np.pi, N_16))


def neighbours_16(p):
    """f(p)_i = p_i p_{i+1}, the indices taken modulo n: the last coordinate is multiplied by the first one."""
    return p * np.roll(p, -1)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M17")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M17, "jacobian_vector_product")

**L'usage** : un produit jacobienne-vecteur sur 2 000 coordonnées, avec et sans la matrice.

In [ ]:
answer_M17_16a = answer_M17_16b = ...
with wbm.attempt("M17.16"):          # if you call your functions here: ⏳ while they are not written
    answer_M17_16a = ...  # a) a number (2 decimals): the norm of J v
    answer_M17_16b = ...  # b) a list of 2 integers: the evaluations of f by jacobian_vector_product, then by
                          #    numerical_jacobian

In [ ]:
wbm.check("M17.16a", answer_M17_16a, computed=True)
wbm.check("M17.16b", answer_M17_16b, computed=True)

## Partie B · Des outils, une prédiction, une expérience et une figure

Fiche §1, §3 et §6. SymPy et SciPy contre ta jacobienne numérique ; la forme et les zéros de la jacobienne d'une composée, prédits puis vérifiés ; la dérivée d'une longue itération ; une grille déformée par une application du plan et par son approximation linéaire. Les vérifications calculent leurs jacobiennes de contrôle avec la `numerical_jacobian` de la référence (`ref_jacobian`), pour ne pas dépendre de la tienne.

In [ ]:
ref_jacobian = wbm.load_mymath("ref", alias="mymath_ref").jacobian   # the reference: the jacobians of the checks

### Ex M17.17 — `sympy.Matrix.jacobian` et `scipy.differentiate.jacobian` contre ta jacobienne numérique 📦 ★ ⏱️ 15 min
**Objectif :** calculer une jacobienne exacte avec SymPy et une jacobienne numérique soignée avec SciPy, et les comparer à ta `numerical_jacobian`.  
**Prérequis :** fiche §1 et §3 ; Ex M17.14 · **Parcours :** C, E

Deux bibliothèques calculent des jacobiennes :
- **SymPy**, exactement : pour une colonne `M = sympy.Matrix([...])` d'expressions, `M.jacobian([r, theta, phi])` rend la jacobienne, une ligne par expression et une colonne par variable (la convention de la fiche) ; `sympy.simplify(...)` simplifie une expression, et `J.subs({r: 2, theta: sympy.pi / 3, phi: sympy.pi / 4})` évalue la matrice en un point (puis `np.array(..., dtype=float)`).
- **SciPy**, numériquement mais avec soin : `scipy.differentiate.jacobian(f, x)` (SciPy 1.15 et après ; importée sous le nom `scipy_jacobian` dans la cellule fournie) rend un objet dont l'attribut `df` est la jacobienne $(m, n)$, `error` une estimation de l'erreur de chaque coefficient, et `nfev` le nombre d'évaluations de $\mathbf{f}$ faites pour chaque coefficient. ⚠️ Pour aller vite, SciPy appelle $\mathbf{f}$ sur **beaucoup de points à la fois** : $\mathbf{f}$ reçoit un tableau de forme $(n, k, \ldots)$, **une colonne par point**, et doit rendre un tableau de forme $(m, k, \ldots)$. Une fonction écrite avec `p[0]`, `p[1]`, `p[2]` et les fonctions de NumPy (`np.sin`, jamais `math.sin`) le fait d'elle-même : c'est le cas de `spherical_17` (cellule fournie), qui sert aussi à ta `numerical_jacobian`.

Les **coordonnées sphériques** : $(r, \theta, \varphi) \mapsto (r\sin\theta\cos\varphi,\ r\sin\theta\sin\varphi,\ r\cos\theta)$, où $\theta$ est l'angle avec l'axe vertical et $\varphi$ l'angle autour de cet axe. Calcule :
- a) avec SymPy et les symboles `r`, `theta`, `phi` de la cellule fournie : le déterminant de la jacobienne, simplifié (une expression SymPy, ou son texte) ;
- b) vrai ou faux : au point $(2 ;\ \frac{\pi}{3} ;\ \frac{\pi}{4})$ (`POINT_17`), la plus grande différence, en valeur absolue, entre ta `numerical_jacobian` et la jacobienne de SymPy évaluée en ce point est sous $10^{-8}$ ;
- c) vrai ou faux : la même chose entre ta `numerical_jacobian` et le `df` de `scipy_jacobian` ;
- d) vrai ou faux : pour **chaque** coefficient, `scipy_jacobian` a fait plus de 6 évaluations de la fonction (son `nfev`), alors que ta `numerical_jacobian` en fait 6 en tout.

Dans ta copie : écris la jacobienne, et retrouve le déterminant à la main (développe selon la 3ᵉ ligne). Que dit $\lvert\det\mathbf{J}\rvert$ du volume de l'image d'une petite boîte $[r, r + \delta r] \times [\theta, \theta + \delta\theta] \times [\varphi, \varphi + \delta\varphi]$ (fiche §1) ? Pourquoi s'annule-t-il sur l'axe vertical, et en $r = 0$ ? Compare l'erreur estimée par SciPy (`error`) à l'écart entre ta jacobienne et celle de SymPy : que gagne SciPy avec ses évaluations en plus ?

In [ ]:
from scipy.differentiate import jacobian as scipy_jacobian

r, theta, phi = sympy.symbols("r theta phi")          # the symbols of a): keep these names
POINT_17 = np.array([2.0, math.pi / 3, math.pi / 4])   # the point (r, θ, φ) of b) to d)


def spherical_17(p):
    """(r, θ, φ) ↦ (r sin θ cos φ, r sin θ sin φ, r cos θ): p is one point, of shape (3,), or an array (3, k, ...) of
    points in columns, as scipy.differentiate.jacobian passes them."""
    return np.array([p[0] * np.sin(p[1]) * np.cos(p[2]), p[0] * np.sin(p[1]) * np.sin(p[2]), p[0] * np.cos(p[1])])

In [ ]:
answer_M17_17a = answer_M17_17b = answer_M17_17c = answer_M17_17d = ...
answer_M17_17a = ...  # a) a SymPy expression of r, theta, phi (or its text): the simplified determinant
with wbm.attempt("M17.17"):          # numerical_jacobian: ⏳ while it is not written
    answer_M17_17b = ...  # b) True or False
    answer_M17_17c = ...  # c) True or False
    answer_M17_17d = ...  # d) True or False

In [ ]:
wbm.check("M17.17a", answer_M17_17a)
for letter, answer in zip("bcd", [answer_M17_17b, answer_M17_17c, answer_M17_17d]):
    wbm.check(f"M17.17{letter}", answer, computed=True)

### Ex M17.18 — Prédire la forme et les zéros de la jacobienne d'une composée, puis vérifier 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la forme et les coefficients nuls de la jacobienne d'une composée à partir de celles de ses facteurs, puis le vérifier en trois points.  
**Prérequis :** fiche §3 · **Parcours :** C, E

Deux applications : $\mathbf{f}(\mathbf{x}) = (x_1x_2,\ x_3^2,\ x_4 + x_1)$, de $\mathbb{R}^4$ dans $\mathbb{R}^3$, et $\mathbf{g}(\mathbf{y}) = (y_1 + y_2,\ y_3^2)$, de $\mathbb{R}^3$ dans $\mathbb{R}^2$ (cellule fournie : `f_18`, `g_18` et `composite_18` $= \mathbf{g} \circ \mathbf{f}$). Raisonne sur les formes, et sur les cases toujours nulles de $\mathbf{J}_{\mathbf{f}}$ et de $\mathbf{J}_{\mathbf{g}}$ : par la règle de la chaîne (fiche §3), le coefficient $(i, j)$ de $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}$ est une somme sur les chemins $x_j \to y_k \to$ sortie $i$. Pour chaque question, choisis le numéro de la bonne réponse :
- a) la forme de $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}$ : (1) $(4, 2)$ ; (2) $(2, 4)$ ; (3) $(3, 3)$ ;
- b) le nombre de coefficients de $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}$ qui sont nuls en **tout** point $\mathbf{x}$ : (1) 0 ; (2) 5 ; (3) 3 ;
- c) la sortie 2 de $\mathbf{g} \circ \mathbf{f}$ dépend-elle de $x_3$ ? (1) oui ; (2) non.

**Avant d'exécuter quoi que ce soit**, écris tes trois prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience : la jacobienne numérique de $\mathbf{g} \circ \mathbf{f}$ (celle de la référence) en trois points tirés au hasard (graine 5031), et les cases nulles aux trois points.

In [ ]:
def f_18(x):
    """f(x) = (x1 x2, x3², x4 + x1), of R^4 into R^3."""
    return np.array([x[0] * x[1], x[2] ** 2, x[3] + x[0]])


def g_18(y):
    """g(y) = (y1 + y2, y3²), of R^3 into R^2."""
    return np.array([y[0] + y[1], y[2] ** 2])


def composite_18(x):
    """g ∘ f, of R^4 into R^2."""
    return g_18(f_18(x))

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M17_18a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M17_18b = ...  # b) 1, 2 or 3
prediction_M17_18c = ...  # c) 1 or 2

In [ ]:
for letter, prediction in zip("abc", [prediction_M17_18a, prediction_M17_18b, prediction_M17_18c]):
    wbm.check(f"M17.18{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M17_18a, prediction_M17_18b, prediction_M17_18c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    rng_18 = np.random.default_rng(5031)
    zero_18 = None
    for x_18 in rng_18.normal(size=(3, 4)):              # three points of R^4 drawn at random
        jacobian_18 = ref_jacobian.numerical_jacobian(composite_18, x_18)
        print(f"en x = {np.round(x_18, 3).tolist()} : forme {jacobian_18.shape}, J =", np.round(jacobian_18, 3).tolist())
        zero_18 = (np.abs(jacobian_18) < 1e-9) if zero_18 is None else zero_18 & (np.abs(jacobian_18) < 1e-9)
    print("cases nulles aux trois points (True) :", zero_18.tolist(), "; il y en a", int(zero_18.sum()))

**Après l'expérience**, dans ta copie : écris $\mathbf{J}_{\mathbf{f}}(\mathbf{x})$, de forme $(3, 4)$, et $\mathbf{J}_{\mathbf{g}}(\mathbf{y})$, de forme $(2, 3)$, puis leur produit, avec $\mathbf{y} = \mathbf{f}(\mathbf{x})$. Pour chaque case toujours nulle du produit, dis quel chemin $x_j \to y_k \to$ sortie $i$ manque. Une case du produit peut-elle être nulle en tout point alors que les termes de sa somme ne le sont pas ? La case $(1, 3)$ est nulle en certains points seulement : lesquels, et pourquoi ne compte-t-elle pas dans b) ?

### Ex M17.19 — Une longue composition : une dérivée qui s'éteint ou explose selon la pente au point fixe 🔬 ★★ ⏱️ 20 min
**Objectif :** suivre la dérivée d'une longue itération par la règle de la chaîne, et relier sa vitesse d'extinction ou d'explosion à la pente au point fixe.  
**Prérequis :** fiche §6 ; R5 · **Parcours :** C

On itère $\varphi_c(x) = cx(1 - x)$ sur $[0 ; 1]$ à partir de $x_0 = 0{,}3$ : $x_{k+1} = \varphi_c(x_k)$, donc $x_k$ est la composée de $k$ copies de $\varphi_c$, appliquée à $x_0$. Par la règle de la chaîne (fiche §6), sa dérivée par rapport au départ est le produit des dérivées le long de la trajectoire :
$$D_k = \frac{\mathrm{d}x_k}{\mathrm{d}x_0} = \prod_{j < k}\varphi_c'(x_j), \qquad D_0 = 1, \qquad D_{k+1} = \varphi_c'(x_k)\,D_k.$$
Écris `iterate_with_derivative(phi, dphi, x0, n_steps)`, qui rend deux tableaux de longueur `n_steps + 1` : les $x_k$ et les $D_k$, de $k = 0$ à `n_steps` (la cellule fournie donne $\varphi_c$ et $\varphi_c'$ par `parabola_19(c)`).

Pour $1 < c < 3$, la trajectoire converge vers le point fixe $x^* = 1 - \frac{1}{c}$, où la pente vaut $\varphi_c'(x^*) = 2 - c$ : les derniers facteurs du produit valent presque $2 - c$, et $D_k$ s'éteint comme $\lvert 2 - c\rvert^k$. Pour $c = 4$, le point fixe $x^* = 0{,}75$ est **repoussant** (pente $-2$) : la trajectoire ne s'en approche pas, elle erre dans $[0 ; 1]$, et $D_k$ grandit en moyenne comme $2^k$, alors que les $x_k$ restent dans $[0 ; 1]$. Calcule, avec $L = 60$ itérations :
- a) pour $c = 2{,}6$, la pente $\varphi_c'(x^*)$ au point fixe (un nombre) ;
- b) pour $c = 2{,}6$, $\lvert D_{60}\rvert^{1/60}$ (3 décimales) ;
- c) pour $c = 4$, le taux moyen de croissance $\frac{1}{60}\ln\lvert D_{60}\rvert$, en **moyenne sur les 10 000 départs** `STARTS_19` (2 décimales) ;
- d) vrai ou faux : pour $c = 4$, les trajectoires parties de $0{,}3$ et de $0{,}3 + 10^{-10}$ sont à plus de $0{,}01$ l'une de l'autre après 40 itérations.

Pourquoi une moyenne en c) : pour $c = 4$, les erreurs d'arrondi des flottants sont, elles aussi, multipliées par 2 à chaque pas en moyenne ; après une cinquantaine d'itérations, la trajectoire calculée n'est plus celle qu'on calculerait exactement, et le taux d'un seul départ dépend de ces détails (d'une écriture de $\varphi_c$ à une autre, par exemple). La moyenne sur beaucoup de départs, elle, n'en dépend presque plus. Les 10 000 départs de `STARTS_19` (cellule fournie) sont plus serrés près de 0 et de 1 : ils sont répartis comme les points d'une très longue trajectoire de $\varphi_4$, ce qui rend la moyenne indépendante des premiers pas. Ta fonction peut prendre un `x0` qui est un tableau de départs, si tu l'écris avec des opérations de NumPy ; sinon, fais une boucle sur les départs (moins d'une seconde).

Une cellule trace ensuite, avec ta fonction, $\lvert D_k\rvert$ en échelle logarithmique pour $c = 1{,}4$ ; $2{,}6$ ; $2{,}8$ et $4$, avec, en pointillés, des droites de pente $\ln\lvert 2 - c\rvert$ pour $c < 3$.

In [ ]:
X0_19, N_STEPS_19 = 0.3, 60
C_VALUES_19 = (1.4, 2.6, 2.8, 4.0)           # the values of c of the figure
STARTS_19 = np.sin(np.pi / 2 * np.linspace(0.0, 1.0, 10002)[1:-1]) ** 2    # the 10 000 starts of c), in ]0, 1[,
                                             # spread as the points of a long trajectory of φ_4 (closer near 0 and 1)


def parabola_19(c):
    """The map φ_c(x) = c x (1 − x) and its derivative φ_c′(x) = c (1 − 2x), for numbers or arrays."""
    return (lambda x: c * x * (1 - x)), (lambda x: c * (1 - 2 * x))

In [ ]:
def iterate_with_derivative(phi, dphi, x0, n_steps):
    """The arrays (x_0, ..., x_n) and (D_0, ..., D_n), n = n_steps: x_{k+1} = phi(x_k), D_0 = 1 and
    D_{k+1} = dphi(x_k) D_k, the derivative dx_k/dx_0 by the chain rule."""
    raise NotImplementedError("iterate_with_derivative n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M17_19a = answer_M17_19b = answer_M17_19c = answer_M17_19d = ...
answer_M17_19a = ...  # a) a number: the slope φ_c′(x*) at the fixed point, c = 2.6
with wbm.attempt("M17.19"):          # if you call iterate_with_derivative here: ⏳ while it is not written
    answer_M17_19b = ...  # b) a number (3 decimals): |D_60|^(1/60), c = 2.6
    answer_M17_19c = ...  # c) a number (2 decimals): the mean over STARTS_19 of ln|D_60| / 60, c = 4
    answer_M17_19d = ...  # d) True or False

In [ ]:
wbm.check("M17.19a", answer_M17_19a)
for letter, answer in zip("bcd", [answer_M17_19b, answer_M17_19c, answer_M17_19d]):
    wbm.check(f"M17.19{letter}", answer, computed=True)

with checked("M17.19 figure"):
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    try:
        steps_19 = np.arange(N_STEPS_19 + 1)
        for c_19, colour_19 in zip(C_VALUES_19, ("C0", "C1", "C2", "C3")):
            xs_19, ds_19 = (np.asarray(a, dtype=float) for a in iterate_with_derivative(*parabola_19(c_19), X0_19,
                                                                                         N_STEPS_19))
            shown_19 = f"{c_19:g}".replace(".", ",")
            ax.semilogy(steps_19, np.abs(ds_19), "-", color=colour_19, lw=1.6, label=f"c = {shown_19}")
            if c_19 < 3:                                     # the slope ln|φ_c′(x*)| = ln|2 − c|, through the last point
                ax.semilogy(steps_19, abs(ds_19[-1]) * abs(2 - c_19) ** (steps_19 - N_STEPS_19), ":", color=colour_19,
                            lw=1.2)
        ax.set_xlabel("itération k")
        ax.set_ylabel("|D_k| = |dx_k / dx_0| (échelle log)")
        ax.set_title("pointillés : la pente ln|2 − c| du point fixe", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Dans ta copie** : pour $c = 2{,}6$, écris les premiers facteurs $\varphi_c'(x_0)$, $\varphi_c'(x_1)$, … : pourquoi b) est-elle un peu plus petite que $\lvert 2 - c\rvert$ ? Pour $c = 2{,}8$, pourquoi la courbe met-elle plus longtemps à rejoindre sa droite, et pourquoi décroît-elle si lentement ? Pour $c = 1{,}4$ et $c = 2{,}6$, les pentes sont les mêmes : qu'est-ce qui distingue les deux trajectoires (regarde le signe des $D_k$) ? Pour $c = 4$, comment l'écart de d) grandit-il entre les itérations 10 et 40, et que dit-il de la prévision de $x_{60}$ à partir d'un $x_0$ connu à $10^{-10}$ près ?

### Ex M17.20 — Une grille du plan déformée par une application non linéaire, et par son approximation linéaire 🎨 ★ ⏱️ 20 min
**Objectif :** voir l'approximation linéaire d'une application du plan : l'image d'une grille par l'application et par $\mathbf{p} \mapsto \mathbf{F}(\mathbf{a}) + \mathbf{J}(\mathbf{p} - \mathbf{a})$, sur un petit et sur un grand carré.  
**Prérequis :** fiche §1 · **Parcours :** C, E

La cellule fournie donne $\mathbf{F}(x, y) = (x + 0{,}4\sin y,\ y + 0{,}4\sin x)$ (`warp_20`, qui prend un point, ou un tableau de forme $(2, k)$ de $k$ points rangés en colonnes), le point $\mathbf{a} = (1 ;\ 0{,}5)$ (`A_20`), et la jacobienne $\mathbf{J}$ de $\mathbf{F}$ en $\mathbf{a}$ (`J_20`, calculée par la `numerical_jacobian` de la référence). Écris `draw_grid_images(ax, F, a, J, half_width, n_lines)`, qui dessine dans l'axe `ax` (sans en ouvrir d'autre) :
- les $n$ lignes horizontales $y = a_y + t$ et les $n$ lignes verticales $x = a_x + t$, pour $t$ dans `np.linspace(-half_width, half_width, n_lines)`, chacune limitée au carré de demi-côté `half_width` centré en $\mathbf{a}$ ;
- l'image de chaque ligne par $\mathbf{F}$, dans une couleur, et par l'approximation linéaire $\mathbf{p} \mapsto \mathbf{F}(\mathbf{a}) + \mathbf{J}(\mathbf{p} - \mathbf{a})$, dans une autre : **une courbe** `ax.plot` par ligne et par application, d'au moins 20 points répartis sur toute la ligne ;
- une légende à deux entrées (un `label` sur une seule courbe de chaque sorte), et des axes de même échelle (`ax.set_aspect("equal")`).

Ne trace pas la grille de départ elle-même, seulement ses deux images. Pour les points d'une ligne rangés en colonnes dans un tableau `P` de forme $(2, k)$, `F(P)` donne leurs images, et `F(a)[:, None] + J @ (P - a[:, None])` celles de l'approximation linéaire.

La cellule de vérification appelle ta fonction deux fois, avec $n = 5$ lignes : sur un petit carré (`half_width = 0.2`) et sur un grand (`half_width = 1.5`). Elle vérifie que chaque courbe tracée est bien l'image d'une ligne de la grille (en remontant de chacun de ses points à son antécédent), qu'il ne manque aucune image, les couleurs et la légende, puis affiche ta figure.

In [ ]:
A_20 = np.array([1.0, 0.5])            # the centre a of the square
N_LINES_20 = 5                          # n lines in each direction


def warp_20(p):
    """F(x, y) = (x + 0.4 sin y, y + 0.4 sin x), for one point (2,) or an array (2, k) of k points in columns."""
    return np.array([p[0] + 0.4 * np.sin(p[1]), p[1] + 0.4 * np.sin(p[0])])


J_20 = ref_jacobian.numerical_jacobian(warp_20, A_20)     # the Jacobian of F at a (the reference's numerical_jacobian)


def preimages_20(points, linear):
    """The points of the plane (one per row) sent onto `points` by F (linear=False), or by p ↦ F(a) + J (p − a)."""
    points = np.asarray(points, dtype=float)
    if linear:
        return A_20 + np.linalg.solve(J_20, (points - warp_20(A_20)).T).T
    p = points.copy()
    for _ in range(80):                    # p = P − 0.4 (sin y, sin x): a contraction of ratio 0.4
        p = points - 0.4 * np.sin(p[:, ::-1])
    return p


def grid_images_20(ax, half_width, n_lines):
    """{(kind, direction, k): colour} for the curves of ax that are the image of a line of the grid: kind "F" or
    "linear", direction "h" (y = a_y + t_k) or "v" (x = a_x + t_k), t_k = np.linspace(-w, w, n)[k]. A curve counts
    when it has at least 20 points, all coming from that line, and covering the whole line."""
    ts = np.linspace(-half_width, half_width, n_lines)
    tolerance = 1e-6 * max(1.0, half_width)
    found = {}
    for line in ax.lines:
        points = np.column_stack([np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)])
        if len(points) < 20 or not np.all(np.isfinite(points)):
            continue
        for kind in ("F", "linear"):
            q = preimages_20(points, kind == "linear") - A_20        # the preimages, from the centre
            for direction, fixed, free in (("h", 1, 0), ("v", 0, 1)):
                k = int(np.argmin(np.abs(ts - q[0, fixed])))
                on_line = bool(np.all(np.abs(q[:, fixed] - ts[k]) < tolerance))
                covers = bool(q[:, free].min() < -0.98 * half_width and q[:, free].max() > 0.98 * half_width
                              and np.all(np.abs(q[:, free]) <= half_width + tolerance))
                if on_line and covers:
                    found[(kind, direction, k)] = mcolors.to_hex(line.get_color())
    return found

In [ ]:
def draw_grid_images(ax, F, a, J, half_width, n_lines):
    """In ax (and no other axes): the images, by F and by the linear approximation p ↦ F(a) + J (p − a), of the
    n_lines horizontal and n_lines vertical lines of the square of half side half_width centred at a; one ax.plot
    curve per line and per map (at least 20 points), one colour per map, a legend."""
    raise NotImplementedError("draw_grid_images n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M17.20"):
    results_20 = []

    def report_20(ok, success, failure):
        results_20.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.0))
    try:
        for ax, width_20 in zip(axes, (0.2, 1.5)):
            shown_20 = f"demi-côté {fr(width_20, 1)}"
            opened_20 = drawn(draw_grid_images, ax, warp_20, A_20.copy(), J_20.copy(), width_20, N_LINES_20)
            report_20(not opened_20, f"{shown_20} : draw_grid_images dessine dans l'axe reçu",
                      f"{shown_20} : draw_grid_images doit dessiner dans l'axe reçu, sans ouvrir de figure (ni "
                      f"plt.figure ni plt.subplots)")
            found_20 = grid_images_20(ax, width_20, N_LINES_20)
            for kind_20, name_20 in (("F", "par F"), ("linear", "par l'approximation linéaire")):
                missing_20 = sum((kind_20, d, k) not in found_20 for d in "hv" for k in range(N_LINES_20))
                report_20(not missing_20, f"{shown_20} : les {2 * N_LINES_20} lignes de la grille ont leur image {name_20}",
                          f"{shown_20} : il manque l'image {name_20} de {missing_20} ligne(s) de la grille sur "
                          f"{2 * N_LINES_20} (une courbe ax.plot par ligne, d'au moins 20 points, qui couvre toute la "
                          f"ligne, du bord du carré à l'autre bord)")
            colours_20 = [{c for (kind, _, _), c in found_20.items() if kind == k} for k in ("F", "linear")]
            if colours_20[0] and colours_20[1]:          # (the missing images are reported above)
                report_20(len(colours_20[0]) == 1 and len(colours_20[1]) == 1 and colours_20[0] != colours_20[1],
                          f"{shown_20} : une couleur par application",
                          f"{shown_20} : une couleur pour toutes les images par F, une autre pour toutes celles de "
                          f"l'approximation linéaire")
            entries_20 = len(ax.get_legend().get_texts()) if ax.get_legend() is not None else 0
            report_20(entries_20 >= 2, f"{shown_20} : une légende",
                      f"{shown_20} : ajoute une légende à deux entrées (label=... sur une courbe de chaque sorte, puis "
                      f"ax.legend()) : j'en compte {entries_20}")
            ax.set_title(shown_20, fontsize=9)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()
    verdict("M17.20", all(results_20), "les images de la grille par F et par l'approximation linéaire sont tracées, "
                                       "pour les deux carrés, en deux couleurs, avec une légende.",
            "relis les lignes ❌ ci-dessus.")

**Regarde ta figure**, puis, dans ta copie : sur le petit carré, les deux grilles sont presque confondues ; sur le grand, elles s'écartent. Comment l'écart entre les deux images d'un coin évolue-t-il quand le demi-côté est multiplié par 7,5 (fiche §1 : de quel ordre est l'erreur de l'approximation linéaire) ? Pourquoi les images par l'approximation linéaire restent-elles des droites parallèles, et celles par $\mathbf{F}$ se courbent-elles ? Écris $\mathbf{J}$ à la main ; que vaut son déterminant, et que dit-il de l'aire de l'image d'un petit carré autour de $\mathbf{a}$ ? Les deux colonnes de $\mathbf{J}$ : où les vois-tu sur la figure ?

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §3, §4 et §5. Trois fonctions fausses à mesurer puis à corriger, un test de propriété de la règle de la chaîne, puis la jacobienne de l'inversion des matrices.

### Ex M17.21 — Chasse à l'erreur : une jacobienne transposée, un produit dans le mauvais ordre, une jacobienne évaluée au mauvais point 🐛 ★ ⏱️ 15 min
**Objectif :** reconnaître trois erreurs classiques de la règle de la chaîne qui ne lèvent aucune erreur Python (une jacobienne transposée, un produit dans le mauvais ordre, un mauvais point d'évaluation), mesurer leurs effets, puis les corriger.  
**Prérequis :** fiche §1, §3 et §4 · **Parcours :** C, E

Un collègue a écrit trois fonctions (cellule fournie) ; leurs docstrings disent ce qu'elles doivent faire, et chacune a l'erreur que nomme le titre :
- `jacobian_by_hand(p)` doit rendre la jacobienne de $\mathbf{f}(x, y) = (x^2,\ x + y)$ au point $\mathbf{p}$ ;
- `chain(Jf, Jg)` doit rendre la jacobienne de $\mathbf{g} \circ \mathbf{f}$ en $\mathbf{x}$, à partir de la jacobienne `Jf` de $\mathbf{f}$ en $\mathbf{x}$ et de la jacobienne `Jg` de $\mathbf{g}$ en $\mathbf{f}(\mathbf{x})$ ;
- `composite_jacobian(f, jac_f, jac_g, x)` doit rendre la jacobienne de $\mathbf{g} \circ \mathbf{f}$ en $\mathbf{x}$, à partir de $\mathbf{f}$ et des fonctions `jac_f` et `jac_g` qui donnent les jacobiennes de $\mathbf{f}$ et de $\mathbf{g}$ en un point ; son erreur : `jac_g` est évaluée au point de départ $\mathbf{x}$ au lieu du point intermédiaire $\mathbf{f}(\mathbf{x})$.

La cellule donne aussi $\mathbf{f}$ (`f_21`), $\mathbf{g}(u, v) = (uv,\ u - v)$ (`g_21`), la jacobienne exacte de $\mathbf{g}$ (`jac_g_21`), une jacobienne numérique de $\mathbf{f}$ (`jac_f_numeric_21`, par la référence) et le point $\mathbf{x} = (1, 2)$ (`X_21`). Aucune des trois fonctions ne lève d'erreur sur ces données. Calcule à la main, puis vérifie en appelant les fonctions du collègue :
- a) ce que rend `chain(jac_f_numeric_21(X_21), jac_g_21(f_21(X_21)))` (une matrice, ses coefficients arrondis à l'entier) ;
- b) ce que rend `composite_jacobian(f_21, jac_f_numeric_21, jac_g_21, X_21)` (une matrice, arrondie à l'entier) ;
- c) la bonne jacobienne de $\mathbf{g} \circ \mathbf{f}$ en $(1, 2)$ (une matrice).

Corrige ensuite les trois fonctions : écris `jacobian_by_hand_fixed`, `chain_fixed` et `composite_jacobian_fixed`, avec les mêmes arguments et les mêmes docstrings. La vérification les essaie sur des cas que tu ne vois pas : d'autres points, d'autres applications, d'autres dimensions.

In [ ]:
def f_21(p):
    """f(x, y) = (x², x + y)."""
    return np.array([p[0] ** 2, p[0] + p[1]])


def g_21(q):
    """g(u, v) = (u v, u − v)."""
    return np.array([q[0] * q[1], q[0] - q[1]])


def jac_g_21(q):
    """The Jacobian of g at q, exact (checked): one row per output."""
    return np.array([[q[1], q[0]],
                     [1.0, -1.0]])


def jac_f_numeric_21(p):
    """The Jacobian of f at p, by the reference's numerical_jacobian."""
    return ref_jacobian.numerical_jacobian(f_21, p)


X_21 = np.array([1.0, 2.0])


# The colleague's three functions
def jacobian_by_hand(p):
    """The Jacobian of f(x, y) = (x², x + y) at p = (x, y), written by hand."""
    x, y = p
    return np.array([[2 * x, 1.0],
                     [0.0, 1.0]])


def chain(Jf, Jg):
    """The Jacobian of g ∘ f at x, from the Jacobian Jf of f at x and the Jacobian Jg of g at f(x)."""
    return Jf @ Jg


def composite_jacobian(f, jac_f, jac_g, x):
    """The Jacobian of g ∘ f at x, from f and from the functions jac_f and jac_g that give the Jacobians of f and g."""
    x = np.asarray(x, dtype=float)
    return jac_g(x) @ jac_f(x)

In [ ]:
answer_M17_21a = ...  # a) a matrix (list of rows)
answer_M17_21b = ...  # b) a matrix
answer_M17_21c = ...  # c) a matrix

In [ ]:
for letter, answer in zip("abc", [answer_M17_21a, answer_M17_21b, answer_M17_21c]):
    wbm.check(f"M17.21{letter}", answer)

**Corrige maintenant les trois fonctions.** La cellule qui suit la tienne les vérifie sur des cas que tu ne vois pas.

In [ ]:
def jacobian_by_hand_fixed(p):
    """The Jacobian of f(x, y) = (x², x + y) at p = (x, y), written by hand."""
    raise NotImplementedError("jacobian_by_hand_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def chain_fixed(Jf, Jg):
    """The Jacobian of g ∘ f at x, from the Jacobian Jf of f at x and the Jacobian Jg of g at f(x)."""
    raise NotImplementedError("chain_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def composite_jacobian_fixed(f, jac_f, jac_g, x):
    """The Jacobian of g ∘ f at x, from f and from the functions jac_f and jac_g that give the Jacobians of f and g."""
    raise NotImplementedError("composite_jacobian_fixed n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M17.21"):
    results_21 = []

    def report_21(ok, success, failure):
        results_21.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def outcome_21(call):
        """(the value of call(), "") or (None, the error it raises); NotImplementedError goes through: ⏳."""
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                return call(), ""
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - reported below
            return None, f"{type(exc).__name__} : {str(exc)[:120]}"

    def same_21(got, want):
        try:
            return got is not None and np.shape(got) == np.shape(want) and bool(np.allclose(np.asarray(got, dtype=float),
                                                                                           want))
        except (TypeError, ValueError):
            return False

    def shown_21(got, error):
        if error:
            return f"ta fonction lève {error}"
        try:
            return "reçu None" if got is None else f"reçu {np.round(np.asarray(got, dtype=float), 3).tolist()}"
        except (TypeError, ValueError):
            return f"reçu {got!r}"[:120]

    rng_21 = np.random.default_rng(5036)
    wrong_points_21 = []                                         # (A) the Jacobian of f at four other points
    for p_21 in rng_21.integers(-4, 5, size=(4, 2)).astype(float):
        want_21 = np.array([[2 * p_21[0], 0.0], [1.0, 1.0]])
        got_21, error_21 = outcome_21(lambda: jacobian_by_hand_fixed(p_21.copy()))
        if not same_21(got_21, want_21):
            wrong_points_21.append(f"en {p_21.tolist()} : attendu {want_21.tolist()}, {shown_21(got_21, error_21)}")
    report_21(not wrong_points_21, "jacobian_by_hand_fixed : la jacobienne de f en quatre autres points",
              "jacobian_by_hand_fixed, " + " ; ".join(wrong_points_21[:2]))
    for n_21, p_21, m_21 in ((2, 3, 4), (2, 2, 2), (3, 1, 2)):  # (B) f of R^n into R^p, g of R^p into R^m
        jf_21 = rng_21.integers(-3, 4, size=(p_21, n_21)).astype(float)
        jg_21 = rng_21.integers(-3, 4, size=(m_21, p_21)).astype(float)
        got_21, error_21 = outcome_21(lambda: chain_fixed(jf_21.copy(), jg_21.copy()))
        report_21(same_21(got_21, jg_21 @ jf_21), f"chain_fixed : f de R^{n_21} dans R^{p_21}, g de R^{p_21} dans R^{m_21}",
                  f"chain_fixed, f de R^{n_21} dans R^{p_21} et g de R^{p_21} dans R^{m_21} : attendu la jacobienne de "
                  f"g ∘ f, de forme ({m_21}, {n_21}) ; {shown_21(got_21, error_21)}")
    cases_21 = [                                                 # (C) label, f, jac_f, jac_g, x
        ("f de R² dans R³, g de R³ dans R²",
         lambda p: np.array([p[0] * p[1], np.sin(p[0]), p[1] ** 2]),
         lambda p: np.array([[p[1], p[0]], [np.cos(p[0]), 0.0], [0.0, 2 * p[1]]]),
         lambda q: np.array([[1.0, q[2], q[1]], [-2 * q[0], 0.0, 1.0]]),            # g(q) = (q1 + q2 q3, q3 − q1²)
         np.array([0.7, -1.3])),
        ("f et g de R² dans R²",
         lambda p: np.array([p[0] + p[1] ** 2, p[0] * p[1]]),
         lambda p: np.array([[1.0, 2 * p[1]], [p[1], p[0]]]),
         lambda q: np.array([[np.cos(q[0]), 0.0], [q[1], q[0]]]),                   # g(q) = (sin q1, q1 q2)
         np.array([0.5, -1.2])),
        ("f de R³ dans R², g de R² dans R",
         lambda p: np.array([p[0] * p[2], p[1] - p[2]]),
         lambda p: np.array([[p[2], 0.0, p[0]], [0.0, 1.0, -1.0]]),
         lambda q: np.array([[2 * q[0] * q[1], q[0] ** 2]]),                       # g(q) = q1² q2
         np.array([1.5, -0.5, 2.0])),
    ]
    for label_21, f_c21, jac_f_c21, jac_g_c21, x_c21 in cases_21:
        want_21 = jac_g_c21(f_c21(x_c21)) @ jac_f_c21(x_c21)
        got_21, error_21 = outcome_21(lambda: composite_jacobian_fixed(f_c21, jac_f_c21, jac_g_c21, x_c21.copy()))
        report_21(same_21(got_21, want_21), f"composite_jacobian_fixed : {label_21}",
                  f"composite_jacobian_fixed, {label_21} : attendu {np.round(want_21, 3).tolist()} ; "
                  f"{shown_21(got_21, error_21)}")
    verdict("M17.21", all(results_21), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque fonction du collègue, que calcule-t-elle exactement, et quelle règle de la fiche viole-t-elle (une ligne par sortie ; la dernière application à gauche ; la jacobienne de $\mathbf{g}$ au point intermédiaire) ? En quels points `jacobian_by_hand` donnerait-elle quand même la bonne matrice ? Pour quelles applications `composite_jacobian` aurait-elle raison (pense à une $\mathbf{g}$ dont la jacobienne ne dépend pas du point) ?

### Ex M17.22 — Un test de propriété : J_{g∘f} = J_g J_f sur des applications tirées au hasard 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété de la règle de la chaîne, qui ne connaît aucune jacobienne exacte, et s'assurer qu'il attrape des implémentations fausses.  
**Prérequis :** fiche §3 ; Ex M17.14 · **Parcours :** C

Écris `test_chain_rule(jacobian)`, une fonction de test dans le style de pytest (des `assert` avec un message qui dit le cas), qui reçoit une fonction `jacobian(f, x)` de même contrat que `numerical_jacobian` (une matrice $(m, n)$, une ligne par sortie) et vérifie la règle de la chaîne du §3 :
- avec un générateur de graine 5035 (`np.random.default_rng(5035)`, créé dans le test), tire une matrice $\mathbf{A}$ de forme $(4, 3)$ et une matrice $\mathbf{B}$ de forme $(2, 4)$, de coefficients de loi normale, et définis $\mathbf{f}(\mathbf{x}) = \sin(\mathbf{A}\mathbf{x})$, de $\mathbb{R}^3$ dans $\mathbb{R}^4$, et $\mathbf{g}(\mathbf{y}) = (\mathbf{B}\mathbf{y}) \odot (\mathbf{B}\mathbf{y})$, de $\mathbb{R}^4$ dans $\mathbb{R}^2$ ;
- en **au moins 3 points** $\mathbf{x}$ tirés au hasard, calcule avec `jacobian` les trois jacobiennes $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}(\mathbf{x})$, $\mathbf{J}_{\mathbf{g}}(\mathbf{f}(\mathbf{x}))$ et $\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ ;
- vérifie d'abord leurs formes, $(2, 3)$, $(2, 4)$ et $(4, 3)$, puis que $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}(\mathbf{x})$ et $\mathbf{J}_{\mathbf{g}}(\mathbf{f}(\mathbf{x}))\,\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ coïncident à une tolérance de l'ordre de $10^{-6}$ (`np.allclose(..., rtol=1e-6, atol=1e-6)`).

C'est un test de **propriété** : il ne compare à aucune formule exacte de jacobienne, seulement trois jacobiennes calculées par la fonction testée.

La vérification appelle ton test sur la `numerical_jacobian` de la référence, où il doit **passer**, et compte les jacobiennes qu'il calcule ; puis sur deux implémentations fausses de la cellule fournie, où il doit **échouer** (par un `assert`, pas par une autre erreur) : `jacobian_transposed`, qui rend la transposée, et `jacobian_forward`, des différences avant avec $h = 10^{-2}$.

Dans ta copie : quel `assert` attrape chacune des deux fausses jacobiennes ? Pourquoi un test de la règle de la chaîne peut-il attraper `jacobian_forward`, alors qu'il ne connaît aucune jacobienne exacte (les erreurs de $\mathbf{J}_{\mathbf{g} \circ \mathbf{f}}$ et de $\mathbf{J}_{\mathbf{g}}\mathbf{J}_{\mathbf{f}}$ se compensent-elles) ? Une jacobienne fausse pourrait-elle passer ce test ? Pense à une fonction qui multiplie toute jacobienne par 2, ou qui rend toujours la matrice nulle de la bonne forme.

In [ ]:
def jacobian_transposed(f, x, h=1e-5):
    """A wrong Jacobian: the transpose of the right one, one row per INPUT."""
    return ref_jacobian.numerical_jacobian(f, x, h).T


def jacobian_forward(f, x, h=1e-2):
    """A wrong Jacobian: forward differences (f(x + h e_j) − f(x)) / h with h = 1e-2, wrong by about h."""
    x = np.array(x, dtype=float)
    f0 = np.atleast_1d(np.asarray(f(x.copy()), dtype=float))
    columns = []
    for j in range(len(x)):
        step = np.zeros_like(x)
        step[j] = h
        columns.append((np.atleast_1d(np.asarray(f(x + step), dtype=float)) - f0) / h)
    return np.column_stack(columns)


WRONG_22 = {"jacobian_transposed (la transposée)": jacobian_transposed,
            "jacobian_forward (des différences avant, h = 10⁻²)": jacobian_forward}

In [ ]:
def test_chain_rule(jacobian):
    """J_{g∘f}(x) = J_g(f(x)) J_f(x), shapes first, for f(x) = sin(A x) of R^3 into R^4 and g(y) = (B y) ⊙ (B y) of R^4
    into R^2 (A and B drawn with the seed 5035), at a few points: three Jacobians computed by `jacobian`."""
    raise NotImplementedError("test_chain_rule n'est pas encore écrit (04_indices.md si tu bloques)")

In [ ]:
with checked("M17.22"):
    if len(inspect.signature(test_chain_rule).parameters) != 1:
        raise TypeError("ton test prend un seul argument, la jacobienne à tester : def test_chain_rule(jacobian)")
    results_22, shapes_22 = [], []

    def report_22(ok, success, failure):
        results_22.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def recorded_22(f, x, h=1e-5):
        """The reference numerical_jacobian, which records the shape of each Jacobian it returns."""
        jacobian = ref_jacobian.numerical_jacobian(f, x, h)
        shapes_22.append(tuple(jacobian.shape))
        return jacobian

    try:
        test_chain_rule(recorded_22)                   # your test on a right Jacobian: ⏳ while it is not written
        why_22 = ""
    except AssertionError as exc:
        why_22 = str(exc)[:200] or "un assert sans message"
    report_22(not why_22, "ton test passe sur la bonne jacobienne numérique",
              f"ton test échoue sur la jacobienne numérique de la référence : il refuse une implémentation juste "
              f"({why_22})")
    counts_22 = {shape: shapes_22.count(shape) for shape in ((2, 3), (2, 4), (4, 3))}
    report_22(min(counts_22.values()) >= 3,
              f"ton test calcule les jacobiennes de g ∘ f, de g et de f en {min(counts_22.values())} points",
              "ton test doit calculer, en au moins 3 points et avec la fonction reçue, les trois jacobiennes : celle "
              "de g ∘ f (forme (2, 3)), celle de g en f(x) (forme (2, 4)) et celle de f (forme (4, 3)) ; j'en compte "
              + ", ".join(f"{n} de forme {s}" for s, n in counts_22.items()))
    hints_22 = {"jacobian_transposed": "vérifie d'abord, par un assert, la forme de chaque jacobienne : (4, 3) pour f, "
                                       "(2, 4) pour g, (2, 3) pour g ∘ f",
                "jacobian_forward": "compare J_{g∘f} à J_g J_f avec une tolérance de l'ordre de 10⁻⁶ "
                                    "(np.allclose(..., rtol=1e-6, atol=1e-6)) : des différences avant avec h = 10⁻² se "
                                    "trompent d'environ 10⁻²"}
    for label_22, wrong_22 in WRONG_22.items():
        try:
            test_chain_rule(wrong_22)
            report_22(False, "", f"ton test passe sur {label_22} : il ne la voit pas ; {hints_22[wrong_22.__name__]}")
        except AssertionError:
            report_22(True, f"ton test échoue sur {label_22}", "")
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - a crash is not a failed assert
            report_22(False, "", f"ton test plante sur {label_22} ({type(exc).__name__} : {str(exc)[:100]}) au lieu "
                                 f"d'échouer sur un assert : {hints_22[wrong_22.__name__]}")
    verdict("M17.22", all(results_22), "ton test accepte la bonne jacobienne, calcule les trois jacobiennes en "
                                       "plusieurs points et attrape les deux fausses.", "relis les lignes ❌ ci-dessus.")

### Ex M17.23 — Défi : la jacobienne de l'inversion X ↦ X⁻¹, vue comme une application de ℝ^{n²} dans ℝ^{n²} 🏆 ★★★ ⏱️ 25 min
**Objectif :** dériver l'inversion des matrices, la voir comme une application de $\mathbb{R}^{n^2}$ dans $\mathbb{R}^{n^2}$, et trouver une formule de sa jacobienne vérifiée numériquement.  
**Prérequis :** fiche §3 et §5 ; M12c · **Parcours :** C

L'inversion $X \mapsto X^{-1}$ des matrices $n \times n$ inversibles est une application de $\mathbb{R}^{n^2}$ dans $\mathbb{R}^{n^2}$, si l'on range les $n^2$ coefficients **ligne par ligne**, comme `X.reshape(-1)` (la convention de NumPy) : la coordonnée $ni + j$ est $X[i, j]$, les indices partant de 0, et de même pour $X^{-1}$. La cellule fournie l'écrit (`inversion_23`) et donne $X_0 = \begin{pmatrix} 2 & 1 \\ 0 & 1 \end{pmatrix}$ (`X0_23`), de déterminant 2 et d'inverse $X_0^{-1} = \begin{pmatrix} 0{,}5 & -0{,}5 \\ 0 & 1 \end{pmatrix}$, et $H = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ (`H_23`). Trouve :
- a) la jacobienne $4 \times 4$ de l'inversion en $X_0$ (4 décimales) ;
- b) son déterminant (une fraction irréductible entre guillemets) ;
- c) la dérivée de l'inversion en $X_0$ dans la direction $H$, la limite de $\frac{(X_0 + tH)^{-1} - X_0^{-1}}{t}$ quand $t \to 0$ (une matrice $2 \times 2$) ;
- d) **l'objectif chiffré** : écris `inverse_jacobian(X)`, la jacobienne $(n^2, n^2)$ de l'inversion en une matrice inversible $X$ de taille $n \times n$, par une **formule** (ni différences finies ni `numerical_jacobian`). La vérification la compare à la jacobienne numérique de la référence sur trois matrices $3 \times 3$ tirées au hasard (graine 5040), décalées de $3\mathbf{I}$ pour être inversibles : le plus grand écart doit rester **sous $10^{-6}$**.

Pour a), ta `numerical_jacobian` et `inversion_23` suffisent. Pour c) et d), dérive l'égalité $X X^{-1} = \mathbf{I}$ : si $X$ bouge de $tH$, comment $X^{-1}$ doit-il bouger pour que le produit reste $\mathbf{I}$, au premier ordre en $t$ ? Puis passe d'une direction à la matrice : la colonne $nk + l$ de la jacobienne est la dérivée dans la direction $E_{kl}$, la matrice qui a un 1 en $(k, l)$ et des 0 ailleurs, aplatie comme les autres.

La cellule de vérification compare tes valeurs de a) à c), puis calcule l'écart de d) et enregistre si l'objectif est atteint.

Dans ta copie : écris ta formule de la dérivée dans une direction $H$, et vérifie c) avec elle. Pour $n = 1$, que dit-elle (la dérivée de $x \mapsto \frac{1}{x}$) ? Retrouve la première colonne de a) à partir d'elle, avec $H = E_{00}$ (le 1 en haut à gauche ; les indices partent de 0, comme dans NumPy). Comment b) s'exprime-t-il à partir de $\det X_0$ ? Pourquoi le déterminant d'une matrice $4 \times 4$ ne change-t-il pas quand on change le signe de tous ses coefficients ?

In [ ]:
X0_23 = np.array([[2.0, 1.0],
                  [0.0, 1.0]])
H_23 = np.array([[0.0, 1.0],
                 [1.0, 0.0]])


def inversion_23(v):
    """X ↦ X⁻¹ as a map of R^(n²) into R^(n²): v is X flattened row by row (X.reshape(-1)), n = √len(v)."""
    n = math.isqrt(len(v))
    return np.linalg.inv(np.reshape(v, (n, n))).reshape(-1)

In [ ]:
def inverse_jacobian(X):
    """The (n², n²) Jacobian at X of X ↦ X⁻¹, the matrices flattened row by row (X.reshape(-1)), by a formula:
    neither finite differences nor numerical_jacobian."""
    raise NotImplementedError("inverse_jacobian n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M17_23a = answer_M17_23b = answer_M17_23c = ...
with wbm.attempt("M17.23"):          # numerical_jacobian: ⏳ while it is not written
    answer_M17_23a = ...  # a) a matrix (4 decimals): the 4 × 4 Jacobian at X0_23
    answer_M17_23b = ...  # b) an irreducible fraction, as text: its determinant
    answer_M17_23c = ...  # c) a matrix: the derivative at X0_23 in the direction H_23

In [ ]:
for letter, answer in zip("abc", [answer_M17_23a, answer_M17_23b, answer_M17_23c]):
    wbm.check(f"M17.23{letter}", answer)

with checked("M17.23d"):
    rng_23 = np.random.default_rng(5040)
    gaps_23 = []
    for draw_23 in range(3):                           # three 3 × 3 matrices, moved by 3 I to be invertible
        X_23 = rng_23.normal(size=(3, 3)) + 3 * np.eye(3)
        want_23 = ref_jacobian.numerical_jacobian(inversion_23, X_23.reshape(-1))
        got_23 = inverse_jacobian(X_23.copy())
        if got_23 is None or np.shape(got_23) != (9, 9):
            print(f"❌ Ex M17.23 : pour une matrice 3 × 3, la jacobienne de l'inversion a la forme (9, 9) ; "
                  f"inverse_jacobian rend {'None' if got_23 is None else 'la forme ' + str(np.shape(got_23))}")
            raise StopCheck
        gaps_23.append(float(np.max(np.abs(np.asarray(got_23, dtype=float) - want_23))))
    print(f"plus grand écart avec la jacobienne numérique, sur les trois matrices : {max(gaps_23):.1e} "
          f"(objectif : moins de 1e-6)")
    wbm.check("M17.23d", bool(max(gaps_23) < 1e-6), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la jacobienne d'une application de $\mathbb{R}^n$ dans $\mathbb{R}^m$, une ligne par sortie, et t'en servir comme approximation linéaire ?
2. Sais-tu reconnaître une jacobienne diagonale, et la calculer avec deux évaluations ?
3. Sais-tu dériver une composée par le produit des jacobiennes, dans le bon ordre et au bon point ?
4. Sais-tu passer du gradient en ligne au gradient de la forme de la variable, et calculer un produit jacobienne-vecteur sans former la matrice ?
5. Sais-tu dériver une fonction définie par morceaux de chaque côté d'un coin, suivre la dérivée d'une longue itération, et calculer une loi de Boltzmann ?

**La suite** : le chapitre 17 de workbookIA, qui se sert de ces outils, puis M18 (différentiation automatique et calcul matriciel), juste avant le chapitre 18 de workbookIA.